# Transit Profiles

`TransitProfile` is the core abstraction for informal transit. It encodes
*typical-speed assumptions* for a mode and time-of-day context — no GTFS
required. This notebook shows every way to construct and use one.

**What's covered**
1. The five named presets and their parameters
2. Overriding individual parameters on a preset
3. Fully custom profiles for one-off cities or modes
4. Time-of-day context: how rush hour shrinks reachable area
5. Coverage type: road-network vs fixed-routes and why it matters
6. Computing area as a function of speed — a sensitivity analysis


## 1 · Named presets

In [ ]:
from idd_travel_time.profiles import list_presets, get_preset
from idd_travel_time.constants import TransitCoverage

print(f"{'Preset':<22} {'Speed (kph)':<16} {'Coverage':<16} Midpoint")
print("-" * 72)
for name in list_presets():
    p = get_preset(name)
    lo, hi = p.typical_speed_kph
    print(f"{name:<22} {lo:4.0f} – {hi:4.0f} kph   {p.coverage.value:<16} {p.midpoint_speed_kph:.1f} kph")


## 2 · Overriding individual parameters

Every parameter is overridable. The most common override is tightening the
speed range for congested urban conditions.


In [ ]:
from idd_travel_time.models import TransitProfile
from idd_travel_time.constants import TimeContext

# Default informal_minibus
default = TransitProfile.from_preset("informal_minibus")

# Rush-hour Lagos: danfo stuck in traffic on Lagos-Ibadan Expressway
rush_lagos = TransitProfile.from_preset(
    "informal_minibus",
    typical_speed_kph=(8.0, 15.0),
    time_context=TimeContext.RUSH_HOUR,
    mode_name="Danfo — rush hour (Lagos)",
)

# Late-night: roads clear, drivers push the limit
night = TransitProfile.from_preset(
    "informal_minibus",
    typical_speed_kph=(20.0, 35.0),
    time_context=TimeContext.NIGHT,
    mode_name="Danfo — night",
)

for p in [default, rush_lagos, night]:
    print(f"{p.mode_name:<35} midpoint={p.midpoint_speed_kph:.1f} kph  context={p.time_context.value}")


## 3 · Fully custom profile

In [ ]:
from idd_travel_time.constants import TransitCoverage, TimeContext

# Electric tuk-tuk pilot in Addis Ababa — faster than a standard bajaj,
# uses road network freely (not restricted to fixed routes)
addis_electric = TransitProfile(
    typical_speed_kph=(20.0, 30.0),
    coverage=TransitCoverage.ROAD_NETWORK,
    time_context=TimeContext.OFF_PEAK,
    mode_name="Electric tuk-tuk (Addis Ababa pilot)",
)

print(addis_electric)
print(f"Midpoint: {addis_electric.midpoint_speed_kph:.1f} kph")
print(f"Coverage: {addis_electric.coverage.value}")


## 4 · Rush hour vs off-peak — area comparison

Same origin, same duration, two profiles. Shows how congestion shrinks
the reachable area.


In [ ]:
from idd_travel_time.models import TravelTimeRequest
from idd_travel_time.constants import TransportMode
from idd_travel_time.isochrone import compute
import geopandas as gpd

origin = dict(origin_lat=-1.2864, origin_lon=36.8172)  # Nairobi CBD

def area_km2(result):
    proj = result.isochrone_gdf.to_crs(result.isochrone_gdf.estimate_utm_crs())
    return proj.geometry.area.iloc[0] / 1e6

profiles = {
    "Night":      TransitProfile.from_preset("informal_minibus", typical_speed_kph=(20.0, 35.0), time_context=TimeContext.NIGHT),
    "Off-peak":   TransitProfile.from_preset("informal_minibus"),
    "Rush hour":  TransitProfile.from_preset("informal_minibus", typical_speed_kph=(8.0, 15.0), time_context=TimeContext.RUSH_HOUR),
}

results = {}
for label, profile in profiles.items():
    req = TravelTimeRequest(
        **origin,
        mode=TransportMode.TRANSIT,
        duration_minutes=30,
        transit_profile=profile,
    )
    results[label] = compute(req)
    print(f"{label:<12}  {area_km2(results[label]):.2f} km²  ({profile.midpoint_speed_kph:.1f} kph midpoint)")


### Overlay the three polygons on one map

In [ ]:
import folium

m = folium.Map(location=[-1.2864, 36.8172], zoom_start=12, tiles="CartoDB positron")
colors = {"Night": "#2ca02c", "Off-peak": "#1f77b4", "Rush hour": "#d62728"}

for label, result in results.items():
    folium.GeoJson(
        result.isochrone_gdf.__geo_interface__,
        name=f"{label} (30 min matatu)",
        style_function=lambda _, c=colors[label]: {
            "color": c, "fillColor": c, "fillOpacity": 0.18, "weight": 2
        },
    ).add_to(m)

folium.Marker([-1.2864, 36.8172], tooltip="Origin").add_to(m)
folium.LayerControl().add_to(m)
m


## 5 · Coverage type: road-network vs fixed-routes

`ROAD_NETWORK` transit (boda-boda, danfo, tuk-tuk) can go anywhere a
vehicle can drive. `FIXED_ROUTES` transit (formal bus, metro) is
constrained to its line network.

The routing backends use this distinction when choosing the OSM network
type — `ROAD_NETWORK` uses the full graph; `FIXED_ROUTES` would ideally
use only transit edges (r5py / GTFS, deferred). For now both use the
driving graph, but the distinction is preserved for when schedule-exact
routing is added.


In [ ]:
for name in list_presets():
    p = get_preset(name)
    icon = "🛣️ " if p.coverage == TransitCoverage.ROAD_NETWORK else "🚌"
    print(f"{icon} {name:<22} → {p.coverage.value}")


## 6 · Sensitivity analysis — area vs speed

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# How does reachable area scale with midpoint speed?
# Approximate: area ≈ π × (speed × duration / 2)²
# (disk approximation — real isochrones are smaller due to road network)

duration_min = 30
speeds = np.linspace(5, 60, 100)
area_approx = np.pi * ((speeds / 60 * duration_min / 2) ** 2)   # km²

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(speeds, area_approx, color="#1f77b4", linewidth=2)
ax.set_xlabel("Midpoint speed (kph)")
ax.set_ylabel("Approximate reachable area (km²)")
ax.set_title(f"Reachable area vs. speed  ({duration_min}-min isochrone, disk approximation)")
ax.grid(alpha=0.3)

# Mark the five presets
for name in list_presets():
    p = get_preset(name)
    s = p.midpoint_speed_kph
    a = np.pi * ((s / 60 * duration_min / 2) ** 2)
    ax.annotate(name, xy=(s, a), fontsize=8, ha="center",
                xytext=(0, 8), textcoords="offset points")
    ax.scatter([s], [a], zorder=5, s=40)

plt.tight_layout()
plt.show()
